In [ ]:
import os
import shutil
from pathlib import Path

# Auto-discover the path
DATASET_PATH = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "pretrain.py" in files:
        DATASET_PATH = root
        break

if DATASET_PATH is None:
    raise FileNotFoundError("Could not find 'pretrain.py' in any /kaggle/input/ structure. Is the dataset attached securely?")

print(f"✓ Auto-detected SBERTa codebase at: {DATASET_PATH}")

# Copy the core python packages and scripts
# NOTE: If you've uploaded a fixed codebase, this will use YOUR version
# If sberta/ doesn't exist in dataset, copy from input; otherwise skip
if not Path("/kaggle/working/sberta").exists():
    shutil.copytree(f"{DATASET_PATH}/sberta", "/kaggle/working/sberta", dirs_exist_ok=True)
    print("✓ Copied sberta/ from dataset")
else:
    print("✓ Using existing sberta/ (not overwriting)")

if not Path("/kaggle/working/pretrain.py").exists():
    shutil.copy(f"{DATASET_PATH}/pretrain.py", "/kaggle/working/pretrain.py")
    print("✓ Copied pretrain.py from dataset")
else:
    print("✓ Using existing pretrain.py (not overwriting)")

# Bring over root tokenizer files if they have been generated
for f in ["sberta.model", "sberta.vocab", "train_tokenizer.py"]:
    if Path(f"{DATASET_PATH}/{f}").exists():
        shutil.copy(f"{DATASET_PATH}/{f}", f"/kaggle/working/{f}")

# CRITICAL: Copy corpus to local SSD for 2-3x faster I/O
LOCAL_CORPUS = "/kaggle/working/corpus"
if not Path(LOCAL_CORPUS).exists():
    print("🚀 Copying corpus to local SSD (speeds up DataLoader streaming)...")
    shutil.copytree(f"{DATASET_PATH}/corpus", LOCAL_CORPUS)
    corpus_size = sum(f.stat().st_size for f in Path(LOCAL_CORPUS).rglob("*.txt")) / 1e6
    print(f"✓ Copied {corpus_size:.1f} MB to local SSD")
    
    # Clean up: keep only the main corpus file
    print("\n🧹 Cleaning corpus directory (keeping only darija_corpus_clean.txt)...")
    main_corpus = Path(LOCAL_CORPUS) / "darija_corpus_clean.txt"
    for subdir in ["toy", "elner", "archived"]:
        subdir_path = Path(LOCAL_CORPUS) / subdir
        if subdir_path.exists():
            print(f"  Removing: {subdir}")
            shutil.rmtree(subdir_path)
    print(f"✓ Training will use only: {main_corpus.name} ({main_corpus.stat().st_size / 1e6:.1f} MB)")
else:
    print("✓ Corpus already on local SSD")

print(f"\n📊 Environment:")
print(f"  Codebase: {DATASET_PATH}")
print(f"  Corpus:   {LOCAL_CORPUS}")
print(f"  Output:   /kaggle/working/runs")


In [ ]:
# Clean up corpus directory - keep only the main corpus file
import shutil
from pathlib import Path

LOCAL_CORPUS = Path("/kaggle/working/corpus")
main_corpus = LOCAL_CORPUS / "darija_corpus_clean.txt"

if main_corpus.exists():
    print("🧹 Cleaning corpus directory (keeping only darija_corpus_clean.txt)...")
    
    # Remove subdirectories
    for subdir in ["toy", "elner", "archived"]:
        subdir_path = LOCAL_CORPUS / subdir
        if subdir_path.exists():
            print(f"  Removing: {subdir}/")
            shutil.rmtree(subdir_path)
    
    # List remaining files
    remaining = list(LOCAL_CORPUS.rglob("*.txt"))
    print(f"\n📊 Corpus files for training:")
    total_size = 0
    for f in remaining:
        size_mb = f.stat().st_size / 1e6
        total_size += size_mb
        print(f"  ✓ {f.name} ({size_mb:.1f} MB)")
    
    print(f"\n✓ Corpus cleaned! Training will use {len(remaining)} file(s), {total_size:.1f} MB total")
else:
    print(f"❌ Main corpus not found at {main_corpus}")


In [ ]:
# RTX 6000 Pro (94.5 GB VRAM) — optimized for maximum throughput
# Medium config (51M params) is the mathematical sweet spot for 27.6M tokens.
# Massive effective batch size (256 * 2 = 512) guarantees perfectly stable Sinkhorn clustering.
# num-workers 0 is kept to prevent DataLoader deadlocks in Kaggle environments.

cmd = f"""python /kaggle/working/pretrain.py \
    --config medium \
    --corpus-dirs /kaggle/working/corpus \
    --tokenizer-dir {DATASET_PATH}/runs/tokenizer \
    --runs-dir /kaggle/working/runs \
    --run-id sberta-medium-50k \
    --total-steps 50000 \
    --warmup-steps 5000 \
    --batch-size 256 \
    --grad-accum 2 \
    --lr 2e-4 \
    --max-length 512 \
    --checkpoint-every 5000 \
    --log-every 100 \
    --num-workers 0"""

print(f"🚀 RTX 6000 Pro training :\n{cmd}\n")
!{cmd}
